# Notebook 09 ? Pipeline, Automation & Deployment

**Project:** Mirae Asset Digital Platform ? User Analytics  
**Phases covered:** Phase 22 ? Phase 23 ? Phase 24 ? Phase 25 ? Phase 26  
**Objective:** Turn the analytics project into a reproducible, automated, deployable system.  

| Deliverable | What | Where |
|-------------|------|-------|
| `pipeline.py` | Automated processing: master data, segmentation, marketing summary, model metrics | `scripts/pipeline.py` |
| `app.py` | Streamlit dashboard (4 tabs ? RFM + K-Means segmentation) | `app/app.py` |
| `README.md` | Project documentation (27-phase roadmap, correct NB table) | `README.md` |
| `requirements.txt` | Python dependencies for pip install | `requirements.txt` |
| `.gitignore` | CSVs intentionally included for Streamlit Cloud deployment | `.gitignore` |

| Phase | Focus | Output |
|-------|-------|--------|
| 22 | Data Pipeline Architecture | System diagram |
| 23 | Power BI Dashboard | Single-file `.pbix` dashboard for clean submission |
| 24 | Automation Pipeline | `pipeline.py` |
| 25 | Streamlit Deployment | `app.py` |
| 26 | Documentation | `README.md`, `requirements.txt`, `.gitignore` |



## Table of Contents

**Phase 22 — Architecture**  
22.1 [System Overview](#271)  
22.2 [Data Flow Diagram](#272)  

**Phase 24 — Automation Pipeline**  
24.1 [Pipeline Design](#201)  
24.2 [Generate pipeline.py](#202)  
24.3 [Test Run](#203)  

**Phase 25 — Streamlit Dashboard**  
25.1 [Dashboard Design](#211)  
25.2 [Launch Instructions](#213)  

**Phase 26 — Documentation**  
26.1 [Project Summary](#222)  


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')
import os, shutil
from pathlib import Path


def find_project_root(start=None):
    current = Path(start or os.getcwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'data').exists() and (candidate / 'notebooks').exists():
            return str(candidate)
    return str(current.parent if current.name == 'notebooks' else current)

BASE = find_project_root()

user_data    = pd.read_csv(os.path.join(BASE,'data','processed','user_data.csv'),
                            parse_dates=['signup_date','last_active_date','first_purchase_date'])
transactions = pd.read_csv(os.path.join(BASE, 'data', 'raw', 'transactions.csv'),
                            parse_dates=['transaction_date'])
transactions = transactions.merge(user_data[['user_id','signup_date']], on='user_id', how='left')
transactions = transactions[transactions['transaction_date'] >= transactions['signup_date']].drop(columns='signup_date')
sessions     = pd.read_csv(os.path.join(BASE,'data','raw','sessions.csv'),
                            parse_dates=['session_date'])
campaigns    = pd.read_csv(os.path.join(BASE,'data','raw','campaigns.csv'),
                            parse_dates=['start_date'])

user_data['churn']           = user_data['churn'].astype(int)
user_data['has_purchased']   = user_data['has_purchased'].astype(int)
user_data['total_sessions']  = user_data['total_sessions'].astype(int)
user_data['total_purchases'] = user_data['total_purchases'].astype(int)

transactions['month']       = transactions['transaction_date'].dt.to_period('M')
transactions['month_str']   = transactions['month'].astype(str)
transactions['day_of_week'] = transactions['transaction_date'].dt.day_name()

print(f'Users        : {len(user_data):,}')
print(f'Transactions : {len(transactions):,}')
print(f'BASE         : {BASE}')


Users        : 10,000
Transactions : 15,000
BASE         : C:\Users\HP\Desktop\Mirae Asset major


---
## Phase 22 — Data Pipeline Architecture

> Document the system before writing any code. Architecture thinking shows you understand where your work fits in a real data stack.


### 22.1 System Architecture Overview


In [2]:
print('='*65)
print('  MIRAE ASSET ANALYTICS - SYSTEM ARCHITECTURE')
print('='*65)
print()
print('LAYER 1 - RAW DATA  (data/raw/)')
print('  users.csv        12,500 rows | visitors + 10,000 registered users')
print('  sessions.csv     50,000 rows | valid post-signup sessions')
print('  transactions.csv 15,000 rows | valid post-signup purchase records')
print('  events.csv       90,000 rows | visitor funnel + registered-user product events')
print('  campaigns.csv    50 rows     | channel-aligned marketing spend')
print()
print('LAYER 2 - PROCESSING  (scripts/pipeline.py)')
print('  Validate raw data and temporal logic')
print('  Aggregate session + transaction metrics per user')
print('  Derive features: recency, revenue ratios, engagement score')
print('  Flag churn (30-day inactivity)')
print('  Build RFM + K-Means segmentation')
print('  Build CAC/ROAS marketing summary')
print('  Train tuned GBM churn metric and save project_metrics.json')
print('  Outputs: user_data.csv, user_data_segmented.csv, marketing_summary.csv, project_metrics.json')
print()
print('LAYER 3 - ANALYTICS  (notebooks/)')
print('  NB01-09: 27 phases of analysis')
print()
print('LAYER 4 - PRESENTATION  (app/app.py)')
print('  Streamlit dashboard | localhost:8501')
print('  Tabs: KPI Overview | Revenue | Churn | Segmentation')
print()
print('OPERATING CYCLE')
print('  raw data -> python scripts/pipeline.py -> processed artifacts -> dashboard auto-updates')
print('='*65)


  MIRAE ASSET ANALYTICS - SYSTEM ARCHITECTURE

LAYER 1 - RAW DATA  (data/raw/)
  users.csv        12,500 rows | visitors + 10,000 registered users
  sessions.csv     50,000 rows | valid post-signup sessions
  transactions.csv 15,000 rows | valid post-signup purchase records
  events.csv       90,000 rows | visitor funnel + registered-user product events
  campaigns.csv    50 rows     | channel-aligned marketing spend

LAYER 2 - PROCESSING  (scripts/pipeline.py)
  Validate raw data and temporal logic
  Aggregate session + transaction metrics per user
  Derive features: recency, revenue ratios, engagement score
  Flag churn (30-day inactivity)
  Build RFM + K-Means segmentation
  Build CAC/ROAS marketing summary
  Train tuned GBM churn metric and save project_metrics.json
  Outputs: user_data.csv, user_data_segmented.csv, marketing_summary.csv, project_metrics.json

LAYER 3 - ANALYTICS  (notebooks/)
  NB01-09: 27 phases of analysis

LAYER 4 - PRESENTATION  (app/app.py)
  Streamlit dashbo

### 22.2 Data Flow Diagram


In [3]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis('off')

# ---------- FUNCTIONS ----------
def draw_box(ax, x, y, w, h, title, subtitle, fc, ec='white', fs=9):
    r = plt.Rectangle((x, y), w, h, facecolor=fc, edgecolor=ec,
                      linewidth=1.5, zorder=2, alpha=0.92)
    ax.add_patch(r)
    
    ax.text(x + w/2, y + h*0.62, title,
            ha='center', va='center',
            fontsize=fs, fontweight='bold', color='white')
    
    if subtitle:
        ax.text(x + w/2, y + h*0.28, subtitle,
                ha='center', va='center',
                fontsize=max(fs-2, 7), color='white', alpha=0.88)

def draw_arrow(ax, x1, y1, x2, y2, color='#555'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.8),
                zorder=1)

# ---------- LAYER LABELS ----------
for ly, ltxt in [(6.6,'LAYER 1: RAW DATA'),
                 (4.9,'LAYER 2: PIPELINE'),
                 (2.9,'LAYER 3: ANALYTICS'),
                 (0.6,'LAYER 4: APP')]:
    ax.text(0.1, ly, ltxt, fontsize=7,
            color='#7f8c8d', fontweight='bold')

# ---------- RAW DATA ----------
raw = [
    (1.2,6.4,1.4,0.75,'users.csv','10K rows'),
    (2.8,6.4,1.6,0.75,'sessions.csv','50K rows'),
    (4.6,6.4,1.8,0.75,'transactions','15K rows'),
    (6.6,6.4,1.5,0.75,'events.csv','90K rows'),
    (8.3,6.4,1.6,0.75,'campaigns','50 rows')
]

for bx,by,bw,bh,bt,bs in raw:
    draw_box(ax, bx, by, bw, bh, bt, bs, '#2980b9', '#1a6fa3')

# ---------- PIPELINE ----------
draw_box(ax,1.2,4.8,8.7,0.90,'pipeline.py',
         'validate | feature eng | segmentation | CAC/ROAS | model metrics | save',
         '#16a085','#0d7a65',fs=10)

# arrows RAW -> pipeline (aligned vertically = cleaner)
raw_centers = [1.9,3.6,5.5,7.35,9.1]
for x in raw_centers:
    draw_arrow(ax, x, 6.4, x, 5.7, '#2980b9')

# ---------- USER DATA ----------
draw_box(ax,3.8,3.65,4.4,0.80,'processed artifacts',
         'user_data | segmented | marketing | metrics','#8e44ad','#6c3483')

draw_arrow(ax, 6.0, 4.8, 6.0, 4.45, '#16a085')

# ---------- ANALYTICS  ----------
nb_info = [
    (0.8,'NB03','EDA insights','#c0392b'),
    (2.2,'NB04','Funnel/Churn/cohort','#e74c3c'),
    (3.6,'NB05','Marketing/cac/revenue','#d35400'),
    (5.0,'NB06','Segmentation','#e67e22'),
    (6.4,'NB07','Modelling','#f39c12'),
    (7.8,'NB08','Business_Strategy','#27ae60'),
    (9.2,'NB09','Deployment','#2ecc71')
]

for bx,bt,bs,bc in nb_info:
    draw_box(ax, bx, 2.55, 1.3, 0.75, bt, bs, bc)

# CLEAN ARROWS (fan style, no overlap chaos)
targets = [1.45,2.85,4.25,5.65,7.05,8.45,9.85]

for tx in targets:
    draw_arrow(ax, 6.0, 3.65, tx, 3.25, '#8e44ad')

# ---------- STREAMLIT ----------
draw_box(ax,2.5,0.45,7.0,1.0,'Streamlit Dashboard  (app/app.py)',
         'KPI Overview | Revenue | Churn | Segmentation | localhost:8501',
         '#27ae60','#1e8449',fs=10)

draw_arrow(ax, 6.0, 2.55, 6.0, 1.45, '#27ae60')

# ---------- TITLE ----------
ax.set_title('Mirae Asset Analytics — End-to-End Data Pipeline Architecture',
             fontsize=13, fontweight='bold', pad=10)

plt.tight_layout()
plt.show()


---
## Phase 23 ? BI Dashboard Export

> The processed CSV layer is intentionally dashboard-ready. Power BI or any BI tool can connect directly to `data/processed/user_data.csv`, `user_data_segmented.csv`, and `marketing_summary.csv` without rerunning notebook logic.


### 23.1 Power BI Export Layer

The Power BI dashboard is packaged as a single submission file: `powerbi/Mirae_Asset_Analytics.pbix`. It was built from processed project outputs, not from separate manual calculations.

**Source files used:**
- `data/processed/user_data_segmented.csv` for user, churn, segment, cluster, revenue, device, state, and monthly summaries.
- `data/processed/marketing_summary.csv` for CAC, ROAS, LTV:CAC, and payback metrics.
- `data/processed/project_metrics.json` for headline KPI values.
- `data/raw/events.csv` for the sequential funnel summary.

**Power BI dashboard inputs used during build:**
- `users.csv` keeps the user-level analytical table for drilldowns.
- `kpi_summary.csv` stores executive KPI values.
- `funnel_summary.csv` stores visit, signup, add-to-cart, and purchase counts/drop-off.
- `monthly_revenue.csv` stores revenue by signup month.
- `channel_summary.csv` stores acquisition-channel performance.
- `marketing_summary.csv` stores CAC, ROAS, LTV:CAC, and payback by channel.
- `risk_summary.csv` stores churn-risk tier metrics.
- `segment_summary.csv` stores RFM segment metrics.
- `cluster_summary.csv` stores K-Means cluster metrics.
- `state_revenue.csv` and `device_summary.csv` support geography/device visuals.

The final handoff is the single `.pbix` file. This avoids exposing the internal PBIP `visual.json` project files in the submitted repository.





---
## Phase 24 - Automation Pipeline

> `pipeline.py` rebuilds all production artifacts in one command. Run it whenever raw data updates. The notebooks and dashboard then read from the same processed source of truth.


### 24.1 Pipeline Design


In [4]:
for step, desc in [
    ('Step 1','Load raw CSVs'),
    ('Step 2','Validate nulls, duplicate user keys, known user IDs, and temporal logic'),
    ('Step 3','Aggregate session metrics per user'),
    ('Step 4','Aggregate transaction metrics per user'),
    ('Step 5','Derive tenure, recency, revenue ratios, and has_purchased'),
    ('Step 6','Compute engagement score and churn flag'),
    ('Step 7','Build RFM segments and K-Means buyer clusters'),
    ('Step 8','Build marketing CAC/ROAS summary'),
    ('Step 9','Compute tuned GBM churn model metrics'),
    ('Step 10','Save user_data.csv, user_data_segmented.csv, marketing_summary.csv, project_metrics.json'),
]:
    print(f'  {step}: {desc}')


  Step 1: Load raw CSVs
  Step 2: Validate nulls, duplicate user keys, known user IDs, and temporal logic
  Step 3: Aggregate session metrics per user
  Step 4: Aggregate transaction metrics per user
  Step 5: Derive tenure, recency, revenue ratios, and has_purchased
  Step 6: Compute engagement score and churn flag
  Step 7: Build RFM segments and K-Means buyer clusters
  Step 8: Build marketing CAC/ROAS summary
  Step 9: Compute tuned GBM churn model metrics
  Step 10: Save user_data.csv, user_data_segmented.csv, marketing_summary.csv, project_metrics.json


### 24.2 Pipeline Source File

The production pipeline is maintained directly in `scripts/pipeline.py`. This notebook audits the pipeline interface and expected functions so deployment documentation stays aligned with the production script.


In [5]:
pipeline_path = os.path.join(BASE, 'scripts', 'pipeline.py')

with open(pipeline_path, 'r', encoding='utf-8') as f:
    pipeline_source = f.read()

print(f'Pipeline path : {pipeline_path}')
print(f'Lines         : {len(pipeline_source.splitlines()):,}')
print(f'Characters    : {len(pipeline_source):,}')
print()
print('Required functions present:')
for fn in [
    'log', 'load_raw_data', 'validate_raw_data', 'build_user_data',
    'assign_rfm_segment', 'build_user_segments', 'build_marketing_summary',
    'build_churn_model_summary', 'build_project_metrics', 'write_outputs', 'run_pipeline'
]:
    print(f'  {fn:28s}: {fn + "(" in pipeline_source}')


Pipeline path : C:\Users\HP\Desktop\Mirae Asset major\scripts\pipeline.py
Lines         : 652
Characters    : 24,653

Required functions present:
  log                         : True
  load_raw_data               : True
  validate_raw_data           : True
  build_user_data             : True
  assign_rfm_segment          : True
  build_user_segments         : True
  build_marketing_summary     : True
  build_churn_model_summary   : True
  build_project_metrics       : True
  write_outputs               : True
  run_pipeline                : True


### 24.3 Test Pipeline Run


In [6]:
import subprocess, sys

# pipeline_path is defined in the cell above (Cell 12)
# Make sure you run Cell 12 before this cell
if 'pipeline_path' not in dir():
    pipeline_path = os.path.join(BASE, 'scripts', 'pipeline.py')

print('Running pipeline.py as a test...')
result = subprocess.run(
    [sys.executable, pipeline_path],
    capture_output=True, text=True, cwd=BASE
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
else:
    ud_test = pd.read_csv(os.path.join(BASE,'data','processed','user_data.csv'))
    print(f'Output: {ud_test.shape[0]:,} rows x {ud_test.shape[1]} cols')
    print(f'Churn rate: {ud_test["churn"].mean():.3f}')
    print('Pipeline test PASSED.')


Running pipeline.py as a test...


[23:49:34] Starting pipeline...
[23:49:34]   Funnel validation: visit=12,500 -> signup=10,000 -> add_to_cart=7,000 -> purchase=4,800
[23:59:23]   User data shape: (10000, 27)  Churn rate: 0.174
[23:59:23]   Segmented shape: (10000, 39)
[23:59:23]   Revenue: Rs 38,095,829  Conversion: 0.480
[23:59:23]   Churn model AUC: 0.826
[23:59:24] Pipeline complete.

Output: 10,000 rows x 27 cols
Churn rate: 0.174
Pipeline test PASSED.


---
## Phase 25 - Streamlit Dashboard

> The Streamlit app is the executive-facing interface: it turns analytical outputs into an interactive dashboard for review, filtering, and decision support.


### 25.1 Dashboard Design


In [7]:
print('Dashboard structure:')
print('  SIDEBAR  : channel | device | gender | churn status | age | phase coverage | model AUC')
print('  TAB 1    : KPI Overview - 6 metric cards + engagement dist + sessions vs revenue scatter')
print('  TAB 2    : Revenue - monthly trend | by channel | by device | by state | Pareto curve')
print('  TAB 3    : Churn - by channel | by age | by device | recency dist | engagement overlay')
print('  TAB 4    : User Segmentation - RFM segment size/revenue/table + K-Means cluster profiles')
print('  ARTIFACTS: dashboard reads user_data.csv, user_data_segmented.csv, and project_metrics.json')


Dashboard structure:
  SIDEBAR  : channel | device | gender | churn status | age | phase coverage | model AUC
  TAB 1    : KPI Overview - 6 metric cards + engagement dist + sessions vs revenue scatter
  TAB 2    : Revenue - monthly trend | by channel | by device | by state | Pareto curve
  TAB 3    : Churn - by channel | by age | by device | recency dist | engagement overlay
  TAB 4    : User Segmentation - RFM segment size/revenue/table + K-Means cluster profiles
  ARTIFACTS: dashboard reads user_data.csv, user_data_segmented.csv, and project_metrics.json


### 25.2 Launch Instructions


In [8]:
print('='*60)
print('  HOW TO LAUNCH THE DASHBOARD')
print('='*60)
print()
print('1. Install Streamlit (once):')
print('   pip install streamlit')
print()
print('2. Run the app:')
print('   cd <project-folder>')
print('   streamlit run app/app.py')
print()
print('3. Open: http://localhost:8501')
print()
print('4. Deploy publicly (free):')
print('   a. Push project folder to GitHub')
print('   b. Go to share.streamlit.io')
print('   c. Connect repo, set main file: app/app.py')
print('   d. Share the live URL with stakeholders')
print('='*60)


  HOW TO LAUNCH THE DASHBOARD

1. Install Streamlit (once):
   pip install streamlit

2. Run the app:
   cd <project-folder>
   streamlit run app/app.py

3. Open: http://localhost:8501

4. Deploy publicly (free):
   a. Push project folder to GitHub
   b. Go to share.streamlit.io
   c. Connect repo, set main file: app/app.py
   d. Share the live URL with stakeholders


---
## Phase 26 - Documentation

> The README is the project brief: it should communicate business context, key findings, and run instructions quickly.


### 26.1 Project Summary


In [9]:
print('='*65)
print('  COMPLETE PROJECT SUMMARY')
print('='*65)
print()

deliverables = [
    ('notebooks/01_data_generation.ipynb',                'Data Generation'),
    ('notebooks/02_data_cleaning_feature_engineering.ipynb', 'Cleaning & Feature Engineering'),
    ('notebooks/03_EDA_Insights_fixed.ipynb',              'EDA'),
    ('notebooks/04_funnel_churn_cohort_analysis.ipynb',    'Funnel / Churn / Cohort'),
    ('notebooks/05_marketing_cac_revenue.ipynb',           'Marketing & Revenue'),
    ('notebooks/06_user_segmentation_clustering.ipynb',    'Segmentation'),
    ('notebooks/07_predictive_modelling_ab_testing.ipynb', 'Modelling & A/B'),
    ('notebooks/08_business_strategy_simulation.ipynb',    'Strategy & Simulation'),
    ('notebooks/09_pipeline_deployment.ipynb',             'Pipeline & Deploy'),
    ('scripts/pipeline.py',                                'Automation script'),
    ('app/app.py',                                         'Streamlit dashboard'),
    ('requirements.txt',                                   'Dependencies'),
    ('.gitignore',                                         'Git config'),
    ('README.md',                                          'Documentation'),
]
print('DELIVERABLES:')
for path, desc in deliverables:
    full = os.path.join(BASE, path.replace('/', os.sep))
    exists = os.path.exists(full)
    size = os.path.getsize(full) if exists else 0
    mark = 'OK' if exists else 'MISSING'
    print(f'  [{mark}] {size:>8,} bytes  {desc}')

processed_files = [
    'user_data.csv',
    'user_data_segmented.csv',
    'marketing_summary.csv',
    'project_metrics.json',
]
print()
print('PROCESSED ARTIFACTS:')
for file in processed_files:
    full = os.path.join(BASE, 'data', 'processed', file)
    exists = os.path.exists(full)
    size = os.path.getsize(full) if exists else 0
    print(f'  [{"OK" if exists else "MISSING"}] {file:28s} {size:>8,} bytes')

print()
print('DATA PIPELINE:')
print(f'  user_data.csv : {len(user_data):,} rows x {len(user_data.columns)} cols')
print(f'  Revenue       : Rs {user_data["total_revenue"].sum():,.0f}')
print(f'  Churn rate    : {user_data["churn"].mean():.3f}')
print(f'  Conversion    : {user_data["has_purchased"].mean():.3f}')
print()
print('COVERAGE: 27 phases | 9 notebooks | Full analytics lifecycle')
print()
print('NEXT STEPS:')
print('  1. Run python scripts/pipeline.py when raw inputs change')
print('  2. streamlit run app/app.py')
print('  3. Review project_metrics.json before updating public claims')
print('  4. Deploy on Streamlit Cloud when ready')
print('  5. Record walkthrough only after metrics and dashboard are final')
print('='*65)


  COMPLETE PROJECT SUMMARY

DELIVERABLES:
  [OK]   34,324 bytes  Data Generation
  [OK]   35,877 bytes  Cleaning & Feature Engineering
  [OK]   47,425 bytes  EDA
  [OK]   97,891 bytes  Funnel / Churn / Cohort
  [OK]   78,818 bytes  Marketing & Revenue
  [OK]   73,712 bytes  Segmentation
  [OK]   75,567 bytes  Modelling & A/B
  [OK]   91,933 bytes  Strategy & Simulation
  [OK]  135,954 bytes  Pipeline & Deploy
  [OK]   24,653 bytes  Automation script
  [OK]   19,769 bytes  Streamlit dashboard
  [OK]      199 bytes  Dependencies
  [OK]      525 bytes  Git config
  [OK]    7,820 bytes  Documentation

PROCESSED ARTIFACTS:
  [OK] user_data.csv                1,702,027 bytes
  [OK] user_data_segmented.csv      2,486,020 bytes
  [OK] marketing_summary.csv           1,164 bytes
  [OK] project_metrics.json            1,257 bytes

DATA PIPELINE:
  user_data.csv : 10,000 rows x 27 cols
  Revenue       : Rs 38,095,829
  Churn rate    : 0.174
  Conversion    : 0.480

COVERAGE: 27 phases | 9 noteboo

---
## Key Findings

**1. The pipeline is the project's production foundation.**  
`pipeline.py` is the single source of truth for processed artifacts. It validates raw data, rebuilds the master dataset, exports segmentation, computes CAC/ROAS, and writes model/project metrics.

**2. Notebook 09 documents the pipeline interface.**  
This notebook validates the expected production functions and explains how the pipeline is used operationally, instead of duplicating implementation logic inside the notebook.

**3. The Streamlit app is the presentation layer, not the calculation layer.**  
The dashboard reads processed artifacts and metrics rather than hard-coding headline values. That keeps public numbers aligned with the pipeline output.

**4. README claims should come from generated metrics.**  
Public claims should be sourced from `data/processed/project_metrics.json` and `marketing_summary.csv`, which keeps the narrative tied to measured outputs.

**5. Architecture thinking closes the loop.**  
The raw data -> pipeline -> analytics -> dashboard flow is reproducible and easy to explain in a business review or walkthrough.
